# @batch vs @kubernetes vs local: execution backends in Metaflow

A practical comparison of the three execution backends Metaflow supports without restructuring a flow. The same `FlowSpec` runs locally by default, can be pushed to AWS Batch with `@batch`, or to a Kubernetes cluster with `@kubernetes`. The point is to see where the compute actually lands, what the cost and latency tradeoffs are, and how to choose per step.

Research note: the comparison is drawn from the Metaflow docs finding that steps "run locally by default; add `@batch` (AWS) or `@kubernetes` to push compute-intensive steps to cloud without restructuring the flow." The runs below need configured cloud infrastructure (AWS Batch / a k8s cluster) to execute end to end; the code shows the pattern and the expected placement behavior.

## The flow

One training flow with three heavy steps. Each step is independently annotated to run on a different backend so the contrast is visible in a single run. In a real project you would usually annotate only the compute-heavy steps and leave the cheap ones local.

In [ ]:
from metaflow import FlowSpec, step, Parameter, resources

class BackendCompareFlow(FlowSpec):

    epochs = Parameter("epochs", help="Training epochs", default=5)

    @step
    def start(self):
        # Lightweight step - keeps it local on purpose.
        self.config = {"lr": 0.01, "epochs": self.epochs}
        print("start: preparing config (local)")
        self.next(self.train_local, self.train_batch, self.train_k8s)

    @step
    def train_local(self):
        # No decorator -> runs on the local machine that launched the flow.
        import time
        time.sleep(1)
        self.backend = "local"
        self.result = sum(range(1000)) / 1000.0
        print(f"train_local: computed on {self.backend}")
        self.next(self.join)

    @resources(cpu=2, memory=4096)
    @step
    def train_batch(self):
        # @batch pushes this step to AWS Batch. Requires a configured Batch
        # environment. Metaflow provisions the container, runs the step, stores
        # the artifact, then tears the compute down.
        self.backend = "aws_batch"
        self.result = sum(range(2000)) / 2000.0
        print(f"train_batch: computed on {self.backend}")
        self.next(self.join)

    @resources(cpu=2, memory=4096)
    @step
    def train_k8s(self):
        # @kubernetes pushes this step to a k8s cluster. Same flow code,
        # different backend. The pod is scheduled, runs the step, and the
        # artifact is stored.
        self.backend = "kubernetes"
        self.result = sum(range(3000)) / 3000.0
        print(f"train_k8s: computed on {self.backend}")
        self.next(self.join)

    @step
    def join(self, inputs):
        # merge_artifacts collects results from the three parallel branches.
        self.results = [i.result for i in inputs]
        self.backends = [i.backend for i in inputs]
        print(f"join: collected results from {self.backends}")
        self.next(self.end)

    @step
    def end(self):
        print("end: all backends finished")

## Running it locally first

Run the whole flow with no cloud config. The `@batch` and `@kubernetes` steps will fail without infrastructure, but `train_local` and the surrounding DAG still validate the logic. This is the cheapest way to develop before paying for cloud compute.

```bash
python 2026-07-19-batch-vs-kubernetes-vs-local.ipynb run
```

Expected: `start` and `train_local` execute locally; `train_batch` / `train_k8s` error unless a Batch / k8s environment is configured. Use this loop to iterate on step logic for free, then switch individual steps to cloud.

## Running the cloud branches

With the Metaflow AWS / kubernetes environment configured, run each annotated step on its backend. The DAG is unchanged - only the decorator decides where the compute lands.

```bash
# Push only the heavy steps to the cloud, keep the rest local.
python 2026-07-19-batch-vs-kubernetes-vs-local.ipynb run --with batch --with kubernetes
```

`--with batch` activates `@batch`; `--with kubernetes` activates `@kubernetes`. Without these flags the decorators are inert and every step stays local.

## Cost / performance tradeoffs

| Backend | Where compute runs | Cold-start cost | Best for | Watch out for |
|---|---|---|---|---|
| local | the machine running the flow | none | dev, tiny datasets, fast iteration | no isolation; blocked by your laptop's resources |
| `@batch` (AWS) | a provisioned AWS Batch job | container + queue spin-up (seconds-minutes) | bursty, long, CPU/GPU-heavy jobs; no cluster to manage | needs Batch + IAM configured; per-job spin-up latency |
| `@kubernetes` | a pod on your k8s cluster | pod scheduling (seconds) | steady workloads, shared cluster, GPU scheduling | you own the cluster; node capacity and quotas matter |

The rule of thumb from the Metaflow docs is sound: keep steps under an hour, and only annotate the compute-intensive ones. Spawning a Batch job for a 2-second step costs more in spin-up than in compute.

One behavior to verify rather than assume: artifact transfer. Metaflow persists `self.result` from each branch and `merge_artifacts` in `join` reassembles them. Confirm in the run's metadata that the three `backend` values appear in the join step's merged artifacts - if a cloud branch never ran, that branch's artifact is simply missing rather than empty.

## Choosing per step

There is no single "best" backend - the choice is per step:

- **Dev loop** -> everything local; iterate until the step logic is correct.
- **One heavy training step** behind a queue, no cluster -> `@batch`.
- **Already on a k8s cluster**, want to share capacity / GPUs -> `@kubernetes`.

The same `FlowSpec` can mix all three, because the backend is a step-level decorator, not a flow-level decision. That is the main reason Metaflow fits cost-performance tradeoffs so well: you pay for cloud only where it earns its keep.

## Verify

After a cloud-enabled run, inspect the run:

```bash
python 2026-07-19-batch-vs-kubernetes-vs-local.ipynb list
```

Then pull the run's artifacts to confirm each branch recorded its backend:

```python
from metaflow import Flow
run = Flow("BackendCompareFlow").latest_successful_run
print(run.data.backends)   # expect ['local', 'aws_batch', 'kubernetes'] when all ran
print(run.data.results)     # three floats, one per branch
```

If `backends` is missing a value, that branch did not execute (likely missing cloud config for that backend).

## Common gotchas

- `@batch` / `@kubernetes` are inert without `--with batch` / `--with kubernetes`, or a configured cloud environment. Locally they just run on your machine.
- `@resources(cpu=, memory=)` is a hint; AWS Batch maps it to a job definition, k8s maps it to pod requests. Mismatched requests can sit pending if the cluster has no fitting node.
- Cloud steps need the same dependencies available at runtime. When using `@pypi` or `@conda`, pin metaflow-compatible packages so the step's container can import them.
- Artifacts flow back to the central metadata store automatically - but only after the step finishes successfully. A crashed cloud step leaves no artifact for the join to merge.